# 05 - Synthèse métier

Objectif : synthétiser les principaux enseignements métier issus de l'analyse et les transformer en recommandations simples.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'incident_event_log.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'data' / 'database'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
incidents = pd.read_csv(PROCESSED_DIR / 'incident_clean.csv')

kpi = pd.DataFrame([{
    'incidents': incidents['number'].nunique(),
    'sla_pct': round(incidents['sla_compliant'].mean() * 100, 2),
    'resolution_moyenne_h': round(incidents['resolution_time_hours'].mean(), 2),
    'cloture_moyenne_h': round(incidents['closure_time_hours'].mean(), 2),
    'reassignations_moyennes': round(incidents['reassignment_count'].mean(), 2),
    'reouvertures_moyennes': round(incidents['reopen_count'].mean(), 2),
}])

def summarize(data, by):
    result = (data.groupby(by, dropna=False)
              .agg(incidents=('number', 'nunique'),
                   sla_pct=('sla_compliant', 'mean'),
                   resolution_moyenne_h=('resolution_time_hours', 'mean'))
              .reset_index())
    result['sla_pct'] = (result['sla_pct'] * 100).round(2)
    result['resolution_moyenne_h'] = result['resolution_moyenne_h'].round(2)
    return result

priority = summarize(incidents, 'priority').sort_values('priority')
groups = (summarize(incidents, 'assignment_group')
          .query('incidents >= 20')
          .nlargest(10, 'incidents'))
monthly = (incidents.groupby('opened_month', dropna=False)
           .agg(incidents=('number', 'nunique'), sla_pct=('sla_compliant', 'mean'))
           .reset_index()
           .sort_values('opened_month'))
monthly['sla_pct'] = (monthly['sla_pct'] * 100).round(2)

incidents['reassignment_status'] = incidents['reassignment_count'].gt(0).map({
    True: 'Avec réassignation', False: 'Sans réassignation'
})
reassignment = summarize(incidents, 'reassignment_status')
long_incidents = (incidents.loc[incidents['resolution_time_hours'].notna(),
                                ['number', 'assignment_group', 'category',
                                 'priority', 'resolution_time_hours']]
                  .nlargest(25, 'resolution_time_hours'))

## Résumé des KPI

In [ ]:
kpi

## Lecture par priorité

In [ ]:
priority

## Lecture par groupe support

In [ ]:
groups

## Lecture des réassignations

In [ ]:
reassignment

## Messages principaux

- Le taux de conformité SLA global est de 63,45 %, ce qui indique un niveau de respect des engagements perfectible.
- Les priorités `1 - Critical` et `2 - High` sont les plus sensibles : leur conformité SLA est inférieure à 3 %, ce qui doit devenir un point de contrôle prioritaire.
- Les réassignations sont associées à des délais de résolution beaucoup plus longs : environ 266 h en moyenne pour les incidents réassignés contre 95 h pour les incidents non réassignés.
- Les groupes support à fort volume doivent être suivis avec le SLA et le délai moyen, pas seulement avec le nombre d'incidents.
- La tendance mensuelle doit être lue avec prudence, car le volume est surtout concentré sur mars, avril et mai 2016.

## Recommandations métier

1. Suivre chaque semaine le taux de conformité SLA global et le détailler par priorité.
2. Mettre en place un suivi spécifique des incidents `Critical` et `High`, car ce sont les catégories les plus risquées pour la qualité de service.
3. Identifier les groupes support qui combinent volume élevé, SLA faible et délais longs.
4. Réduire les réassignations inutiles en améliorant les règles de routage et la qualification initiale des incidents.
5. Surveiller les incidents longs avant qu'ils dépassent fortement les délais attendus.